In [1]:
import sys
import pandas as pd
from datasets import Dataset
import csv

# Increase CSV field size limit (to handle long legal text fields)
csv.field_size_limit(sys.maxsize)

# Now load the CSV
df_full = pd.read_csv("/content/train (1).csv", engine='python', on_bad_lines='skip')
df = df_full[:4000]


In [2]:
# Convert to Hugging Face dataset
dataset = Dataset.from_pandas(df)
dataset

Dataset({
    features: ['Case Name', 'Input', 'Output', 'Label', 'Count', 'Decision_Count'],
    num_rows: 3119
})

In [3]:
df_test = pd.read_csv("/content/test.csv", engine='python', on_bad_lines='skip')
df_test = df_test[:1000]

In [4]:
# Drop rows with missing or non-string Input/Output
df_test = df_test[df_test["Input"].notna() & df_test["Output"].notna()]
df_test = df_test[df_test["Input"].apply(lambda x: isinstance(x, str))]
df_test = df_test[df_test["Output"].apply(lambda x: isinstance(x, str))]

# Then convert to Hugging Face dataset


In [5]:
test = Dataset.from_pandas(df_test)
test

Dataset({
    features: ['Case Name', 'Input', 'Output', 'Label', 'Count', 'Decision_Count', '__index_level_0__'],
    num_rows: 927
})

In [6]:
# Step 1: Initial train/test split (90% train + val, 10% test)
# split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
# train_val = split_dataset["train"]
# test = split_dataset["test"]

# Step 2: Now split train_val into train and validation
train_val_split = dataset.train_test_split(test_size=0.01, seed=42)  # 0.1111 ≈ 500 of 4500
train = train_val_split["train"]
val = train_val_split["test"]


In [7]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

# Measure max input and output lengths
input_lengths = train.map(lambda x: {"len": len(tokenizer(x["Input"])["input_ids"])})["len"]
output_lengths = train.map(lambda x: {"len": len(tokenizer(x["Output"])["input_ids"])})["len"]

max_input_len = min(max(input_lengths), 512)   # Flan-T5 input cap
max_output_len = min(max(output_lengths), 128) # You can raise this if needed

# ✅ FIXED: Tokenization function for batched input
def tokenize(batch):
    return tokenizer(
        batch["Input"],
        text_target=batch["Output"],
        truncation=True,
        padding="max_length",
        max_length=max_input_len
    )

# Apply tokenizer to each dataset split
tokenized_train = train.map(tokenize, batched=True)
tokenized_val = val.map(tokenize, batched=True)
tokenized_test = test.map(tokenize, batched=True)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Token indices sequence length is longer than the specified maximum sequence length for this model (6739 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/3087 [00:00<?, ? examples/s]

Map:   0%|          | 0/3087 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/927 [00:00<?, ? examples/s]

In [8]:
import transformers
print(transformers.__version__)  # Should be >= 4.30


4.52.4


In [9]:
from transformers import TrainingArguments
print(TrainingArguments.__module__)


transformers.training_args


In [10]:
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer

model_name = "google/flan-t5-base"  # or another if you've chosen differently
model = T5ForConditionalGeneration.from_pretrained(model_name)

training_args = TrainingArguments(
    output_dir="./legal_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_strategy="steps",
    logging_steps=10,
    save_total_limit=1,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",  # or another if you're calculating accuracy/rouge
)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [11]:
# from sklearn.metrics import accuracy_score, f1_score

# def compute_metrics(pred):
#   labels = pred.label_ids
#   preds = pred.predictions.argmax(-1)
#   f1 = f1_score(labels, preds, average='weighted')
#   acc = accuracy_score(labels, preds)
#   return {"accuracy": acc, "f1": f1}

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
)

/tmp/ipython-input-12-411242984.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [13]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [14]:
api="5a00d59cae42cfed01d259bdab0b1f83f1533929"

In [15]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bhoot869 (bhoot869-university-system-of-georgia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,2.883100,2.471081
2,2.494500,2.441142


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=3088, training_loss=3.0101500088686772, metrics={'train_runtime': 2316.3214, 'train_samples_per_second': 2.665, 'train_steps_per_second': 1.333, 'total_flos': 4227691894013952.0, 'train_loss': 3.0101500088686772, 'epoch': 2.0})

In [16]:
!ls ./legal_model

checkpoint-3088


In [17]:
print(tokenized_train[0])


{'Case Name': 'Union Of India Vs. West Punjab Factories Ltd', 'Input': 'Wanchoo, J. 1. These three appeals raise common questions and will be dealt with together. They arise out of two suits filed against the Government of India claiming damages for loss of goods which were destroyed by fire on the railway platform at Morar Road Railway Station. One of the suits was filed by Birla Cotton Factory Limited, now represented by the West Punjab Factories Limited (hereinafter referred to as the factory). It related to six consignments of cotton bales booked from six stations on various dates in February and March 1943 by the factory to Morar Road Railway Station. In five of the cases, the consignment was consigned to J. C. Mills while in one it was consigned to self. The consignments arrived at Morar Road Railway Station on various dates in March. Delivery was given of a part of one consignment on March 7, 1943 while the remaining goods were still in the custody and possession of the railway.

In [18]:
# metrics = trainer.evaluate(tokenized_test)
# print(metrics)


In [19]:
def predict_case(facts: str):
    # Tokenize the input
    input_ids = tokenizer(facts, return_tensors="pt").input_ids

    # Get the device of the model
    device = model.device

    # Move the input tensor to the same device as the model
    input_ids = input_ids.to(device)

    # Generate the output
    outputs = model.generate(input_ids, max_new_tokens=300)

    # Decode and return the output
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

sample_input ="""Subba Rao, J.1. These two appeals by special leave are filed against the common judgment of a Full Bench of the High Court of Judicature at Nagpur in writ petitions Nos. 22 of 1955 and 274 of 1955 filed by respondents 1, 3 to 6 herein in the said Court.2. The facts in Appeal No. 370 of 1960 may be stated first. Respondent 1, Seth Balkishan Nathani, was the proprietor and lambardar of Mouza Sonpairi in Tehsil and District Raipur. On January 14, 1947, he executed perpetual pastas in favour of his wife, Yashodabai, since deceased, and respondents 4, 5 and 6 in respect of khudkasht and grass lands of Mouza Sonpairi. In Tabdili Jamabandi of the year 1941347 the said lands were recorded as the Occupancy Tenancy Holdings of the said respondent 4 to 6 and respondent 2, Govindlal Nathani, the legal representative of Yashodabai. The same entry was found in the Jamabandis of the subsequent years. The Madhya Pradesh Abolition of Proprietary Rights (Estates, Mahals, Alienated Lands) Act, l950 (1 of 1951), hereinafter called the Act, came into force on January 22, 1951. Thereafter in due course the estate of the said proprietor was duly notified under S. 3 of the Act. On March 25, 1952, the Deputy Commissioner, Land Reforms, acting under S. 40 of the Act, recognised the said Balkishan Nathani as the pattadar and settled the assessment payable by him in respect of Khasra Nos. 289/2 and 366/7 of Mouza Sonpairi. No appeal was preferred against that order. Thereafter, appellant 2, the Nistar Officer-cum-Additional Deputy Commissioner, Raipur, started proceedings against the respondents for the correction of old annual papers in Mouza Sonpairi with a view to reopen the earlier order made under S. 40 of the Act, as the earlier order was passed on the basis of the entries found in Tabdili Jamabandi of the year 1946-47 and subsequent years. Respondent 1, Seth Balkishan Nathani, raised an objection that appellant 2 had no jurisdiction to initiate the proceedings. Appellant 2 overruled the objection and made the following order:"On the next hearing, 5 witnesses may be produced for proving cultivation. The names of the purchasers, to whom the lands have been sold, be obtained from the Patwari and a notice he served on them that they should file their statements as well as should bring the sale-deeds along with them. Hearing fixed for date 4-8-1954. The non-applicants may file other evidence which they wish to file."It will be seen from the said order that the second appellant purported to make an inquiry in regard to the factum of cultivation as well as the validity of the sale-deeds whereunder respondent I created interests in the other respondents. Respondent 1 preferred an appeal from that order to the Board of Revenue, Madhya Pradesh, but the same was dismissed on the ground that it was premature. Thereupon, the respondents filed the writ petition No. 22 of 1955 in the High Court of Madhya Pradesh.3. Civil Appeal No. 371 of 1960 relates to Patti No. 1 of Mouza Kachna in Tehsil and District Raipur. Respondent 1 was the Proprietor and Lambardar of the said Mouza. On February 19, 1948, the said Seth Balkishan Nathani executed perpetual pastas in respect of the said lands in favour of the same respondents as in the other appeal. In the annual papers the said lands were recorded as the Occupancy Tenancy Holdings of respondents 2 to 6. On December 8, 1954, appellant 2 made an inspection of the said lands and made the following order on December 9, 1954:" * * * *2. There were found to be obvious mistakes in Government documents Khasra, Jamabandi and Tabdilat, Mistakes discharged (discovered) by me in Patwari papers have been corrected.3. Ex-proprietors (I) Balkishan Nathani and others and (2) Narayanrao made absolutely bogus transfers in favour of their family members, namely(i) (a) Kamlabai, (b) Pana Bai, (c) Yashodabai, (d) Chhote Bai of Nathani family.(ii) Kamla Bai Chitnavis, wife of Narayanrao, ax-proprietor.Patwari entered names without cultivation and agricultural possession against Land Record Manual. Volume 1.4. Mistakes found in patwari records have been corrected by me after spot inspection.These papers be now filed."It will be seen from the said order that the second appellant found that the transfers made by respondent l in favour of the other respondents were bogus and that he also corrected the entries in the annual papers to the effect that the landlord was not cultivating the lands as recorded in the earlier papers. The respondents filed writ petition No 274 of 1905 in the High Court to quash the said order. A Full Bench of the High Court held that neither S. 15 (3) of the Act nor S. 47 (1) of the Central Provinces Land Revenue Act, 1917 (C. P. Act No. [I of 1917), hereinafter called the Land Revenue Act, conferred a power on the Nistar Officer to review orders already made in respect of the factum of cultivation or the occupancy rights recognized under the relevant provisions of the said Acts. In the result, it allowed the two writ petitions quashing the proceedings started by the Nistar Officer in the case of Mouza Sonpairi and the order, dated December 9, 1954, passed by him in the case of Mouza Kachna and prohibiting him from taking further proceedings which may affect the occupancy tenancy rights of the petitioners in the lands in dispute. Hence the two appeals.4. Mr. Sen, learned counsel for the appellants, raised before us the following two points: (1) under S. 47 (1) of the land Revenue Act, the Nistar Officer has jurisdiction to correct entries made for earlier years in a subsequent year on the ground of mistake; and (2) the said officer has also jurisdiction to review under S. 15 (3) of the Act the order made by him under S. 40 thereof.5. Mr. Pai, learned counsel for the respondents, argued at the outset that the appeals have abated for two reasons, namely, (1) the second petitioner died after the arguments were heard by the High Court and before the judgment was delivered and the petition filed by the appellants to set aside abatement was dismissed, and (2) the second respondent in the appeals died on March 7, 1956 and the application filed on June 28, 1957 to set aside the abatement and to bring his legal representatives on record was out of time. On the merits, he sought to sustain the judgment of the High Court for the reasons mentioned therein.6. As we are inclined to agree with the view expressed by the High Court on the two questions raised by the learned counsel for the appellants, we do not propose to consider the preliminary objection raised by the learned counsel for the respondents.7. The two questions raised in this case are in a way inter-related and the answer to them depends upon the construction of the relevant sections of the Act and the Land Revenue Act. It would be convenient to read the relevant provisions.The Madhya Pradesh Abolition of Proprietary Rights (Estates, Mahals Alienated Lands) Act, 1950 (Act 1 of l95l).:Section 3. (2) After the issue of a notification under sub-s. (1), no right shall be acquired in or over land to which the said notification relates, except by succession or under a grant or contract in writing made or entered into by or on behalf of the State and no fresh clearings for cultivation or for any other purpose shall he made in such land except in accordance with such rules as may be made by the State Government in this behalf.Section 4. (2) Notwithstanding anything contained in sub-s. (1), the proprietor shark continue to retain the possession of his homestead, home-farm land, and in the Central Provinces also of land brought under cultivation by him after the agricultural year 1948-49 but before the date of vesting.Section 18. (1) On receipt of the statement of claim, or if no such claim is received within the prescribed period, the Compensation Officer shall, after making such enquiry as he thinks fit and giving an opportunity to the claimant to be heard, decide the amount of compensation due to the claimant and record in a statement in the prescribed form the details of the land which shall vest in the State Government after its acquisition in lieu of the payment of such compensation and other details as may be prescribed.Section 15. (1) Any person aggrieved by the decision given or the record made under S. 13 by the Compensation Officer may appeal to the Deputy Commissioner....* * * *3. The Compensation Officer, the Deputy Commissioner or the Settlement Commissioner, may, either on his own motion or on the application filed within the prescribed period by any party interested, review an order passed by himself or his predecessors in office and pass such order in reference thereto as he thinks fit.* * * *Section 40. (as amended on October 22 1951).(1) An, land not included in homefarm but brought under cultivation by the proprietor after the agricultural year 1948-49 shal1 he held by him in the rights of an occupancy tenant.(2) Any person becoming an occupancy tenant under rule I shall be a tenant of the State.(3) The Deputy Commissioner shall determine the rent on the land and it shall be payable from the date of the vesting of the proprietary rights.Section 84. Except where the provisions of this Act provide otherwise. from every decision or order of a Revenue Officer under this Act or the rules made thereunder, an appeal shall lie as if such decision or order has been passed by such officer under the Central Provinces Land Revenue Act, 1917, or the Berar Land Revenue Code, 1928, as the case may be.The Central Provinces Land Revenue Act, 19l7.Section 45. (1) A Record of Right for each mahal or estate shall be prepared or revised, as the case may be, by the Settlement Officer at settlement and for such mahals or estates as the Provincial Government may direct, by a Revenue Officer empowered by the Provincial Government in that behalf during the currency of a settlement.(2) The Record of Rights of a mahal shall consist of the following documents:-(a) Khewat or statement of persons possessing proprietary rights in the mahal, including inferior proprietors or lessees or mortgagees in possession, specifying the nature and extent of the interest of each;(b) Khasra or field-book, in which shall be entered the names of all persons cultivating or occupying land, the right in which it is held, and the rent, if any, payable;(c) Jamabandi or list of persons cultivating or occupying land in the village;* * * *(4) The documents specified in sub-s. (2) shall be prepared in such form and shall contain such additional particulars as may be prescribed by rules made under S. 227.Section 46. On the implication of any person interested therein or of his own motion, the Deputy Commissioner may without prejudice to other provisions of this Act, modify any entry in the Record of Rights on one or more of the following grounds:-(a) that all persons interested in such entry wish to have it modified or(b) that by a decree in a civil suit it has been declared to be erroneous, or(c) that, being founded on a decree or order of a Civil Court or on the order of a Revenue Officer. it is not in accordance With such decree or order; or* * * *Section 47. (1) The Deputy Commissioner shall cause to be prepared, in accordance with rules made under S. 227, for each Mahal, annually or at such longer intervals as may be prescribed, an amended set of the documents mentioned in S. 45, sub-s. (2), Cls. (b), (c) and (d), and the documents so prepared shall be called the "annual papers".(2) The Deputy Commissioner shall cause to be recorded, in accordance with rules made under S. 227, all charges that have taken place in respect of, and all transactions that have affected, any of the proprietary rights and interests in any land.* * * *8. The scheme of Act so far as it is relevant to the present enquiry may be summarized thus: On the issue of a notification by the State Government under S. 3 of the Act in respect of an estate, all proprietary rights in such estate vest in the State. The Compensation Officer, on a claim made by the proprietor, after making the enquiry prescribed under the said Act, decides the amount of compensation due to him and the details of the land that vests in the State. But the Act saves some interests in the proprietor from its total operation: one of such is lands in the Central Provinces brought under cultivation by the proprietor after the agricultural year 1948-49, but before the date of the vesting: (see S. 4 (2) of the Act).9. Under S. 40 (1) of the Act, such a land shall be held by him in the rights of an occupancy tenant; under sub-s. (2) thereof he becomes a tenant of the State, and under sub-s. (3) the Deputy Commissioner shall determine the rent on the land and it shall be payable from the date of the vesting of the proprietary rights. Section 84 confers a right of appeal on an aggrieved party against the order of the Deputy Commissioner to the prescribed authority. There is no provision in the Act which authorizes the Deputy Commissioner to review an order made by him under the said subsection and, therefore, an order made by him, subject to appeal, becomes final. It is therefore, manifest that the order made by the Deputy Commissioner in respect of lands in question determining the rent on the basis that the proprietor was an occupancy tenant had become final. If so, the Nistar Officer, i.e., the second appellant, had no jurisdiction to initiate proceedings for reopening the order made in respect of Mouza Sonpairi or in making the order reviewing the earlier order made by him in respect of Mouza Kachna, for the said orders had become final and there is no provision under the Act for reviewing them. But the learned counsel for the appellants contends that S. 15 (3) of the Act confers such a power. Under S. 15 (3) of the Act, the authority concerned can review an order made by him under S. IS of the Act. Section 13 of the Act deals with an order made by the Compensation Officer deciding the amount of compensation due to the claimant and recording in a statement in the prescribed form the details of the land which shall vest in the State. Neither S. 13 nor S. 15(3) has any relevance in the context of an order made by the Deputy Commissioner under S. 40 of the Act.10. This conclusion would be sufficient to dispose of the appeals. But, as an argument was made on the construction of Section 47 (1) of the Land Revenue Act and as the same was considered by the High Court, we shall also deal with it.11. The argument based upon the said provision is relevant more to the nature of the evidence available to the Deputy Commissioner to come to a decision under S. 40 of the Act than to the validity or the finality of the order made by him thereunder. The question that a Deputy Commissioner has to decide by necessary implication uncle S. 40 of the Act is whether the proprietor has cultivated the land after the agricultural year 1948-49 and before the vesting of the estate in the State. One of the most important pieces of evidence that will be available to him is the annual papers prepared under S. 47 of the Land Revenue Act. It is not disputed that in the annual papers prepared earlier it was shown that the proprietor was cultivating the lands in question after 1948-49. But it is said that under S. 47 (1), the Deputy Commissioner can correct the said entry in the year 1952 and 1954 as he purports to do, so as to make the entry to the effect that between 1949 and the date of the investigation the proprietor was not in cultivation of the land. This argument, if we may say so, is contrary to the scope and tenor of the relevant provisions of the Land Revenue Act and the rules made thereunder. Under Ss. 45, 46 and 47, the provisions whereof we have extracted earlier, the procedure prescribed is as follows: A Record of Rights shall consist of Khewat, Khasra Jamabandi and other papers; and they are prepared in the manner prescribed by the rules made under S. 227. On the application of any person interested therein or of his own motion, the Deputy Commissioner may modify any entry in the Record of Rights on specified grounds, namely, that all persons interested in such entry wish to have it modified, that by decree in a civil suit it has been declared to be erroneous, that, being founded on a decree or order of a civil Court or on the order of a Revenue Officer, it is not in accordance with such decree or order, and that being so founded, such decree or order has subsequently been varied on appeal, revision or review.It will be seen that a mistake in a Khasra or Jamabandi of an earlier year in regard to the factum of cultivation by a particular person is not a ground for modification under S. 46 of the Land Revenue Act. Section 47 empowers the Deputy Commissioner to cause to be prepared annually or at such longer intervals as may prescribed, an amended set of the documents mentioned in Cls. (b), (c) and (d) of sub-s. (2) of S. 45 of the Land Revenue Act, and the documents so prepared shall be called the "annual papers". The rules made under S. 227 of the Land Revenue Act are found in Chap. III of the Central Provinces Land Records Manual Vol. 1, pp. 13-16. The rules relevant to the preparation of Khasra and Jamabandi direct the Patwari to record such changes annually as he finds to have taken place after local enquiry and actual inspection. It is, therefore, clear that a Record of Rights consists of Khewat, Khasra, Jamabandi, etc., and till it is revised again it will hold the field. The entries therein can be modified only for the grounds mentioned in S. 46 of the Land Revenue Act. The provisions of S. 47 if contrasted with those of S. 46, make it clear that the said section intends to bring the said documents up-to-date by recording the subsequent changes based on supervening events. The scope of the annual papers is only to record the existing facts on the basis of spot inspection at the beginning of a fasli and to record changes occurring during the course of the year after the year is closed. It is not the province of the annual papers to investigate and decide on the correctness or otherwise of the entries made in the earlier annual papers as on the date they were made.12. The said section came under judicial scrutiny of a Division Bench of the Nagpur High Court in Mangloo v. Board of Revenue, ILR (1954) Nag 141 (146). The facts in that case were that on the death of one Gaindoo who was a tenant of mouza Matia, on an application made by his nephew and his widow, their names were entered in the annual papers as joint tenants of the land by the Assistant Superintendent of Land Records, thereafter, the widow applied to the Superintendent of Land Records for striking off the petitioners name from the annual papers and her application was allowed; in appeal, the Additional Deputy Commissioner declined to interfere on the ground that the initial order of the Assistant Superintendent of Land Records was passed by him in his executive capacity and as such the Superintendent of Land Records was competent to modify it in his own executive capacity; the second appeal preferred to the Board of Revenue was summarily rejected, and it was contended before the High Court that the decision of the Board of Revenue contravened the provisions of S. 47 (1), read with S. 33 (2) (c) of the Central Provinces Land Revenue Act, 1917. In that context, the learned Judges of the High Court considered the scope of S. 47 (1) of Land Revenue Act and the rules made under S. 227 of the said Act, and observed thus:"As we read S. 47 (1) of the Act and the rules governing it, we are of opinion that these provisions deal only with the preparation of the annual papers and not with their correction if the entries are found to be erroneous. They are only enabling provisions which import no restriction on the power of the Revenue Officers to correct the mistakes or remove any irregularities, committed in the preparation of annual paper. Neither the annual papers nor the corrected entries affect any questions of title or vested interest of any party. The power of the Revenue Officers in this regard is analogous to the untrammelled right of a person to correct his private documents, which cannot be questioned in a Court of law by anyone whose right or interest is not affected thereby."The learned counsel contends that the said passage comprises conflicting ideas inconsistent with each other-the first part of it denying a right to correct the entries and the second part permitting such corrections. We cannot accept this interpretation of the passage. The learned Judges were dealing with two aspects of the question: one is the scope of the preparation of the annual papers and the other is whether correction of mistakes therein give a cause of action to the person aggrieved. The first they answered by stating that S. 47 (1) of the Land Revenue Act and the rules made under the said Act deal only with the preparation of the annual papers and not with their corrections if the entries ale found to be erroneous and the other with the right of a party affected by the correction of the mistakes therein. The observations made in regard to the scope of S. 47 (1) are made clear by the discussion found earlier in the judgment at p. 145. After adverting to the provisions of S. 47 and the rules made under the Act governing the preparation of annual papers the learned Judges observed:"This could normally be clone in the beginning of the agricultural year which, under S. 2 (1) of the Act, commences on the first day of June. No changes in the entries are contemplated during the course of the agricultural year and the changes taking place during that period are obviously to be recorded after the year is closed. The action taken by the Superintendent of Land Records and ratified by the Additional Deputy Commissioner has, therefore, no reference to the preparation of the annual papers under S. 47 (1) of the Act, and we are not shown any other provision of law which governs it."The Division Bench held that there was no provision for correcting the wrong entries made in the annual papers, for their scope is very limited. This view was followed by the Full Bench of the High Court in their judgment which is now under appeal. The Full Bench confirmed the view of the Division Bench in the following words:".......Section 47 (1) of the Central Provinces Land Revenue Act contemplates entering only such changes in the annual papers as take place during the course of the agricultural year. That section, therefore, does not cover a case of correction of the entries on the ground of mistake."We entirely agree with this view. It follows that the Nistar Officer, has no jurisdiction to correct the said entries with a view to reopen the matter already closed under S. 40 of the Act. We, therefore agree with the conclusion arrived at by the High Court."""
print(predict_case(sample_input))

0[ds]We are of opinion that the said passage contains conflicting ideas inconsistent with each other-the first part of it denying a right to correct the mistakes therein. They are only enabling provisions which import no restriction on the power of the Revenue Officers to correct the mistakes or remove any irregularities, committed in the preparation of annual papers. That section, therefore, does not cover a case of correction of the entries on the ground of mistake. We, therefore, agree with the conclusion arrived at by the High Court. We, therefore agree with the conclusion arrived at by the High Court.


In [20]:
# trainer.train(resume_from_checkpoint="./results/checkpoint-1000")


In [21]:
preds = []
labels = []

for example in test:  # use raw test, not tokenized_test
    input_text = example["Input"]
    label_text = example["Output"]

    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True).input_ids
    device = model.device

    # Move the input tensor to the same device as the model
    inputs = inputs.to(device) # Changed from input_ids to inputs
    outputs = model.generate(inputs, max_new_tokens=256) # Changed from input_ids to inputs
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    preds.append(pred_text.strip())
    labels.append(label_text.strip())

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [24]:
!pip install rouge_score' for instance

/bin/bash: -c: line 1: unexpected EOF while looking for matching `''
/bin/bash: -c: line 2: syntax error: unexpected end of file


In [27]:
import evaluate
rouge = evaluate.load("rouge")

rouge_scores = rouge.compute(predictions=preds, references=labels)
print("🔍 ROUGE Scores:")
for key, score in rouge_scores.items():
    print(f"{key}: {score:.4f}")


🔍 ROUGE Scores:
rouge1: 0.2330
rouge2: 0.0779
rougeL: 0.1675
rougeLsum: 0.1673
